# Análise Exploratória e Direcionamento de Modelagem - Classificador de Notícias (AeC)

Este notebook registra a análise exploratória do dataset de notícias em português utilizado na etapa técnica da **AeC Centro de Contatos**.

O foco desta etapa é entender a qualidade da base, caracterizar a variável-alvo e levantar evidências para as decisões de modelagem e validação da API.

### Objetivos desta análise:
1. **Entender a base:** verificar estrutura do CSV, valores ausentes e consistência geral dos dados.
2. **Avaliar risco de viés no treino:** identificar duplicidades e desbalanceamento entre categorias.
3. **Caracterizar os títulos:** observar o tamanho dos textos para apoiar validações de entrada.
4. **Orientar a modelagem:** justificar escolhas de pré-processamento e a adoção de uma baseline clássica para classificação de texto.

---

## 1. Configuração e Importações
A análise foi escrita com a biblioteca padrão do Python para manter a execução simples e leve.

In [ ]:
import csv
import collections
from pathlib import Path

# Caminho do dataset real
DATASET_PATH = Path("../data/raw/articles.csv")

path_to_use = DATASET_PATH
print(f"Utilizando o dataset localizado em: {path_to_use.resolve()}")

## 2. Carga dos Dados e Tratamento de Nulos
Nesta etapa, o arquivo CSV é lido e cada coluna é inspecionada quanto à presença de valores ausentes. Isso ajuda a entender se existe alguma limitação estrutural que possa impactar a análise ou o treino.

In [ ]:
# Analisar os cabeçalhos e ler registros
with open(path_to_use, "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)

print(f"Colunas identificadas no CSV: {header}")

rows = []
null_counts = {col: 0 for col in header}
total_rows_read = 0

with open(path_to_use, "r", encoding="utf-8") as file:
    # O DictReader mapeia automaticamente as linhas baseado no cabeçalho
    reader = csv.DictReader(file)
    for row in reader:
        total_rows_read += 1
        rows.append(row)
        for col in header:
            val = row.get(col, "").strip()
            if not val:
                null_counts[col] += 1

print(f"\nTotal de linhas lidas: {total_rows_read}")
print("Valores ausentes detectados por coluna:")
for col, null_c in null_counts.items():
    pct = (null_c / total_rows_read) * 100 if total_rows_read > 0 else 0.0
    print(f"  - {col}: {null_c} nulos ({pct:.2f}%)")

## 3. Identificação e Remoção de Duplicados
Como o problema é de classificação de títulos, registros duplicados podem inflar métricas e gerar vazamento entre treino e teste. Por isso, a análise mede e elimina duplicidades com base na coluna principal de texto (`title`).

In [ ]:
# Identificar a coluna de texto do título
text_col = "title" if "title" in header else ("titulo" if "titulo" in header else header[0])

vistos = set()
linhas_limpas = []
duplicados = 0

for row in rows:
    texto = row.get(text_col, "").strip()
    if not texto:
        continue  # descartar nulos na coluna principal
    
    if texto in vistos:
        duplicados += 1
    else:
        vistos.add(texto)
        linhas_limpas.append(row)

print(f"Registros únicos preservados: {len(linhas_limpas)}")
print(f"Registros duplicados descartados: {duplicados} ({ (duplicados / total_rows_read) * 100:.2f}% do total)")

## 4. Distribuição das Categorias Alvo
A variável-alvo precisa ser observada antes do treino para verificar concentração excessiva em poucas classes. Esse ponto é importante para a escolha da métrica de avaliação e para o tratamento de classes raras.

In [ ]:
# Identificar a coluna alvo (categoria)
target_col = "category" if "category" in header else ("categoria" if "categoria" in header else header[-1])

category_counts = collections.Counter(row.get(target_col, "").strip() for row in linhas_limpas)

print(f"Total de categorias distintas detectadas: {len(category_counts)}")
print("\nDistribuição das categorias (Top 15):")
sorted_cats = category_counts.most_common(15)
total_limpos = len(linhas_limpas)

for cat, count in sorted_cats:
    pct = (count / total_limpos) * 100
    # Barra gráfica em modo texto para visualização
    bar = "#" * int(pct / 2)
    print(f"  {cat:<18} | {count:>6} ({pct:>5.2f}%) {bar}")

## 5. Análise do Tamanho dos Títulos
O comprimento dos títulos ajuda a calibrar validações mínimas de entrada e a entender o nível de informação disponível em cada amostra, já que o classificador trabalha apenas com o título da notícia.

In [ ]:
tamanhos = [len(row.get(text_col, "").strip()) for row in linhas_limpas]

tamanhos_sorted = sorted(tamanhos)
n = len(tamanhos)
media = sum(tamanhos) / n if n > 0 else 0
mediana = (tamanhos_sorted[n//2] + tamanhos_sorted[-(n//2+1)]) / 2 if n > 0 else 0
minimo = tamanhos_sorted[0] if n > 0 else 0
maximo = tamanhos_sorted[-1] if n > 0 else 0

print(f"Estatísticas de tamanho dos títulos (em caracteres):")
print(f"  - Média:   {media:.2f}")
print(f"  - Mediana: {mediana:.2f}")
print(f"  - Mínimo:  {minimo}")
print(f"  - Máximo:  {maximo}")

## 6. Leituras da EDA para a Modelagem

1. **Desbalanceamento entre categorias:** a base apresenta concentração relevante em algumas classes, o que torna insuficiente avaliar o modelo apenas por acurácia. Por isso, a comparação de candidatos no projeto prioriza **macro F1**.
2. **Controle de duplicidades:** como títulos repetidos distorcem a avaliação, a remoção de duplicados antes do treino é uma medida importante para reduzir vazamento entre conjuntos.
3. **Validação de entrada da API:** a observação dos menores títulos da base sustenta a exigência de um tamanho mínimo de **3 caracteres**, evitando predições sobre entradas muito pobres em sinal.
4. **Pré-processamento textual:** normalização de acentuação, tokenização simples e remoção de stopwords em português são escolhas adequadas para reduzir ruído lexical sem encarecer a solução.
5. **Escolha de baseline:** como o problema trabalha com títulos curtos, uma abordagem clássica com **TF-IDF** e classificadores lineares/probabilísticos oferece uma base sólida, explicável e fácil de produtizar.